In [1]:
print("system_state_devitions")

system_state_devitions


In [2]:
import sys
print(sys.version)

3.10.20 (main, Mar  3 2026, 09:24:47) [GCC 13.3.0]


In [3]:
from pyspark.sql import SparkSession
spark= SparkSession.builder\
       .appName("System_State_definitions")\
       .getOrCreate()
spark
       
  


26/04/10 13:37:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
s_df= spark.read.parquet("/home/ubuntu/thesis/data/processed_data/feature_engineering")
s_df.show()
s_df.count()

#df_active=s_df.filter(s_df.page_count>10)
#df_active.count()


+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|
+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+
|"""European_Socie...|2015-07-01|         0|       0.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|
|"""European_Socie...|2015-07-02|         0|       0.0|       0.0|    0| NULL|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-03|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-04|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-05|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-06|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-07|         0|       0.0|       0.0|    0| 

79783550

In [5]:
s_df.selectExpr(
    "percentile(page_count,0.01)",
    "percentile(page_count,0.05)",
    "percentile(page_count,0.10)",
    "percentile(page_count,0.25)"
).show()

[Stage 7:>                                                          (0 + 1) / 1]

+-------------------------------+-------------------------------+-------------------------------+-------------------------------+
|percentile(page_count, 0.01, 1)|percentile(page_count, 0.05, 1)|percentile(page_count, 0.10, 1)|percentile(page_count, 0.25, 1)|
+-------------------------------+-------------------------------+-------------------------------+-------------------------------+
|                            0.0|                            0.0|                            1.0|                           10.0|
+-------------------------------+-------------------------------+-------------------------------+-------------------------------+



In [6]:
# filtering the pages that have page_count >15 for better evaluation

df_active= s_df.filter(s_df.page_count>=15)
df_active.show(5)
print("the size of active pages count is " ,df_active.count())

+--------------------+----------+----------+----------+----------+-----+-----+--------+---------+-------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|   trend|deviation|z_score|
+--------------------+----------+----------+----------+----------+-----+-----+--------+---------+-------+
|"""Weird_Al""_Yan...|2015-07-01|      2403|    2403.0|      NULL| NULL| NULL|    NULL|      0.0|   NULL|
|"""Weird_Al""_Yan...|2015-07-02|     20136|   11269.5|  12539.12| 2403| NULL| 17733.0|   8866.5|   0.71|
|"""Weird_Al""_Yan...|2015-07-03|      1850|   8129.67|  10401.47|20136| 2403|-18286.0| -6279.67|   -0.6|
|"""Weird_Al""_Yan...|2015-07-04|      1432|    7806.0|  10680.14| 1850|20136|  -418.0|  -6374.0|   -0.6|
|"""Weird_Al""_Yan...|2015-07-05|      1351|   1544.33|     267.8| 1432| 1850|   -81.0|  -193.33|  -0.72|
+--------------------+----------+----------+----------+----------+-----+-----+--------+---------+-------+
only showing top 5 rows



[Stage 9:=====================================================>   (15 + 1) / 16]

the size of active pages count is  57168842


In [7]:
df_inactive=s_df.filter(s_df.page_count<15)
df_inactive.show(5)
print("The size of inactive dataframe page_count is ",df_inactive.count())

+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|
+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+
|"""European_Socie...|2015-07-01|         0|       0.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|
|"""European_Socie...|2015-07-02|         0|       0.0|       0.0|    0| NULL|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-03|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-04|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
|"""European_Socie...|2015-07-05|         0|       0.0|       0.0|    0|    0|  0.0|      0.0|   NULL|
+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+
only showing top 5 rows



[Stage 13:====================================================>   (15 + 1) / 16]

The size of inactive dataframe page_count is  22614705


In [8]:
print(df_inactive.count()/s_df.count())

0.28345072386475656


In [9]:
df_active.show()

+--------------------+----------+----------+----------+----------+-----+-----+--------+---------+-------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|   trend|deviation|z_score|
+--------------------+----------+----------+----------+----------+-----+-----+--------+---------+-------+
|"""Weird_Al""_Yan...|2015-07-01|      2403|    2403.0|      NULL| NULL| NULL|    NULL|      0.0|   NULL|
|"""Weird_Al""_Yan...|2015-07-02|     20136|   11269.5|  12539.12| 2403| NULL| 17733.0|   8866.5|   0.71|
|"""Weird_Al""_Yan...|2015-07-03|      1850|   8129.67|  10401.47|20136| 2403|-18286.0| -6279.67|   -0.6|
|"""Weird_Al""_Yan...|2015-07-04|      1432|    7806.0|  10680.14| 1850|20136|  -418.0|  -6374.0|   -0.6|
|"""Weird_Al""_Yan...|2015-07-05|      1351|   1544.33|     267.8| 1432| 1850|   -81.0|  -193.33|  -0.72|
|"""Weird_Al""_Yan...|2015-07-06|      1792|    1525.0|    234.75| 1351| 1432|   441.0|    267.0|   1.14|
|"""Weird_Al""_Yan...|2015-07-07|      1710|  

In [10]:
#from pyspark.sql.functions import when,abs,col
#df_active_labl = df_active.withColumn(
  #  "system_state_label",
  #  when(abs(col("z_score")) <= 1, "Normal")
   # .when((abs(col("z_score")) > 1) & (abs(col("z_score")) <= 2), "Degraded")
   # .when((abs(col("z_score"))>2 ),"Failure")
#)

#df_active_labl.show()

In [11]:
#print("failure pages are " ,df_active_labl.filter(col("system_state_label") == "Failure").count())
#print("degraded pages are ",df_active_labl.filter(col("system_state_label") == "Degraded").count())
#print("Normal pages are " ,df_active_labl.filter(col("system_state_label") == "Normal").count())

In [12]:
df_active.selectExpr(
    "max(z_score)",
    "min(z_score)"
).show()

[Stage 23:=================================================>      (14 + 2) / 16]

+------------+------------+
|max(z_score)|min(z_score)|
+------------+------------+
|        1.16|       -1.16|
+------------+------------+



In [13]:
 # cahnge the logic and divide into centralized and distributed dara sets here in this note book
#  Normal-- <40% of max Z score
# degraded -- 40% of max <z score <75% of max value 
# failure  --- >75% of max value 

# compute percentile values and 

In [14]:
df_active.select("z_score").describe().show()

[Stage 26:====================================================>   (15 + 1) / 16]

+-------+--------------------+
|summary|             z_score|
+-------+--------------------+
|  count|            57054585|
|   mean|-0.00185483375262...|
| stddev|  0.8529138446236298|
|    min|               -1.16|
|    max|                1.16|
+-------+--------------------+



In [15]:
df_active.select("Moving_std").show()

+----------+
|Moving_std|
+----------+
|      NULL|
|  12539.12|
|  10401.47|
|  10680.14|
|     267.8|
|    234.75|
|    234.55|
|    171.26|
|    171.26|
|    210.72|
|    347.43|
|    193.87|
|    185.58|
|    138.84|
|     27.54|
|     66.83|
|     91.83|
|     184.0|
|     100.6|
|    286.09|
+----------+
only showing top 20 rows



In [16]:
df_active.selectExpr(
    
    "percentile(page_count,0.10) as p10",
    "percentile(page_count,0.25) as p25",
    "percentile(page_count,0.30) as p30",
    "percentile(page_count,0.40) as p40",
    "percentile(page_count,0.50) as p50",
    "percentile(page_count,0.60) as p60",
    "percentile(page_count,0.70) as p70",
    "percentile(page_count,0.80) as p80",
).show()

[Stage 32:>                                                         (0 + 1) / 1]

+----+----+-----+-----+-----+-----+-----+------+
| p10| p25|  p30|  p40|  p50|  p60|  p70|   p80|
+----+----+-----+-----+-----+-----+-----+------+
|29.0|80.0|108.0|185.0|301.0|473.0|735.0|1195.0|
+----+----+-----+-----+-----+-----+-----+------+



In [17]:
# for distributed data set

from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank
from pyspark.sql.functions import avg,when,col
window = Window.partitionBy("Page").orderBy("z_score")
df_distributed = df_active.withColumn(
    "percentile",
    percent_rank().over(window)
)

df_distributed.show()

[Stage 35:>                                                         (0 + 1) / 1]

+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+--------------------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|
+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+--------------------+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|
|"""Weird_Al""_Yan...|2015-08-30|        20|     26.67|      5.77|   30|   30|-10.0|    -6.67|  -1.16|0.001824817518248...|
|"""Weird_Al""_Yan...|2015-08-18|        19|     25.33|      5.51|   28|   29| -9.0|    -6.33|  -1.15|0.003649635036496...|
|"""Weird_Al""_Yan...|2015-09-23|        40|      56.0|     13.89|   65|   63|-25.0|    -16.0|  -1.15|0.003649635036496...|
|"""Weird_Al""_Yan...|2015-10-08|        38|      51.0|     11.27|   58|   57|-20.0|    -13.0|  -1.15|0.003649635036496...|
|"""Weir

In [18]:
df_distributed= df_distributed.withColumn("state_label",
                                     when(col("percentile")<=0.40, "Normal")
                                     .when((col("percentile")> 0.40) & (col("percentile")<= 0.75) , "Degraded")
                                     .when(col("percentile")>0.75 , "Failure")
                                    )
df_distributed.show()

+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|                Page|      Date|page_count|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|
+--------------------+----------+----------+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|"""Weird_Al""_Yan...|2015-07-01|       144|     144.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|
|"""Weird_Al""_Yan...|2015-08-30|        20|     26.67|      5.77|   30|   30|-10.0|    -6.67|  -1.16|0.001824817518248...|     Normal|
|"""Weird_Al""_Yan...|2015-08-18|        19|     25.33|      5.51|   28|   29| -9.0|    -6.33|  -1.15|0.003649635036496...|     Normal|
|"""Weird_Al""_Yan...|2015-09-23|        40|      56.0|     13.89|   65|   63|-25.0|    -16.0|  -1.15|0.003649635036496...|     Normal|
|"""Weird_Al""_Yan...|2015-10-08|        38|    

In [19]:
print("failure pages in distributed dataset are " ,df_distributed.filter(col("state_label") == "Failure").count())
print("degraded pages in distributed dataset are ",df_distributed.filter(col("state_label") == "Degraded").count())
print("Normal pages distributed dataset are " ,df_distributed.filter(col("state_label") == "Normal").count())

failure pages in distributed dataset are  14131005


degraded pages in distributed dataset are  20020173


[Stage 53:=================================================>        (6 + 1) / 7]

Normal pages distributed dataset are  23017664


In [20]:
# for centralized data frame dataset

#df_active.show()

from pyspark.sql.functions import sum
from pyspark.sql.functions import avg,stddev

df_central = df_active.groupBy("Date").agg(
    sum("page_count").alias("total_load")
)
df_central.show(5)



global_mean = df_central.select(avg("total_load")).collect()[0][0]
print("global Mean is ",global_mean)
global_stddev=df_central.select(stddev("total_load")).collect()[0][0]
print("global std is " ,global_stddev)


+----------+------------+
|      Date|  total_load|
+----------+------------+
|2016-03-01|2.00883104E8|
|2016-04-25|2.01316469E8|
|2015-09-02|1.58419933E8|
|2015-12-22|1.64836511E8|
|2016-05-03|1.93639637E8|
+----------+------------+
only showing top 5 rows



global Mean is  189746753.8018182


[Stage 66:====================================================>   (15 + 1) / 16]

global std is  31528631.684786838


In [21]:
df_central.count()

550

In [22]:
from pyspark.sql.functions import col
df_centralized= df_central.withColumn("z_score",
                                      (col('total_load')-global_mean)/(global_stddev)
                                     )
df_centralized.show()
                                      
                                      


[Stage 78:====================================================>   (15 + 1) / 16]

+----------+------------+--------------------+
|      Date|  total_load|             z_score|
+----------+------------+--------------------+
|2016-03-01|2.00883104E8| 0.35321387586748043|
|2016-04-25|2.01316469E8|  0.3669590013880753|
|2015-09-02|1.58419933E8| -0.9935991233306194|
|2015-12-22|1.64836511E8|  -0.790083218671296|
|2016-05-03|1.93639637E8| 0.12347136523721065|
|2016-07-26|3.07642362E8|  3.7393188951827767|
|2016-08-15|3.16771898E8|  4.0288822384725895|
|2016-08-31|2.30883619E8|  1.3047462893237807|
|2016-10-03|2.18026566E8|   0.896956534013728|
|2016-01-28|1.86548428E8|-0.10144194755402103|
|2015-09-28|1.74449321E8|-0.48519177599450003|
|2016-07-17|1.83120557E8|-0.21016442667302487|
|2016-11-08|2.23464939E8|  1.0694465124679506|
|2016-12-19|2.20680241E8|   0.981123681720314|
|2016-07-03|1.87911811E8|-0.05819925267177347|
|2016-08-23| 2.0132483E8|  0.3672241889193196|
|2015-10-16|1.55572607E8| -1.0839083390449789|
|2015-11-02|1.78953304E8|-0.34233803451185724|
|2015-11-27|1

In [23]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag
window = Window.orderBy("Date")
df_centralized=df_centralized.withColumn("lag1",lag("total_load",1).over(window))
df_centralized=df_centralized.withColumn("lag2",lag("total_load",2).over(window))
df_centralized.show()

26/04/10 13:50:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+
|      Date|  total_load|            z_score|        lag1|        lag2|
+----------+------------+-------------------+------------+------------+
|2015-07-01|1.48519499E8|-1.3076131946985547|        NULL|        NULL|
|2015-07-02| 1.4943842E8|  -1.27846759113456|1.48519499E8|        NULL|
|2015-07-03|1.40993628E8|-1.5463127702222008| 1.4943842E8|1.48519499E8|
|2015-07-04|1.45438186E8|-1.4053438235062359|1.40993628E8| 1.4943842E8|
|2015-07-05|1.51332605E8|-1.2183893416584186|1.45438186E8|1.40993628E8|
|2015-07-06|1.60572622E8|-0.9253218501041155|1.51332605E8|1.45438186E8|
|2015-07-07|1.53986154E8| -1.134226190319365|1.60572622E8|1.51332605E8|
|2015-07-08|1.48684975E8|-1.3023647588750664|1.53986154E8|1.60572622E8|
|2015-07-09| 1.4935827E8|-1.2810097249258807|1.48684975E8|1.53986154E8|
|2015-07-10|1.48206312E8|-1.3175466102406925| 1.4935827E8|1.48684975E8|
|2015-07-11|1.45151172E8| -1.414447104703767|1.48206312E8| 1.493

In [24]:
# for moving Average
from pyspark.sql.functions import avg,round
window_MA= Window.orderBy("Date").rowsBetween(-2,0)
df_centralized=df_centralized.withColumn("Mov_Avg",round(avg("total_load").over(window_MA),4))
df_centralized.show(5)

26/04/10 13:50:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:50:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+
|      Date|  total_load|            z_score|        lag1|        lag2|         Mov_Avg|
+----------+------------+-------------------+------------+------------+----------------+
|2015-07-01|1.48519499E8|-1.3076131946985547|        NULL|        NULL|    1.48519499E8|
|2015-07-02| 1.4943842E8|  -1.27846759113456|1.48519499E8|        NULL|   1.489789595E8|
|2015-07-03|1.40993628E8|-1.5463127702222008| 1.4943842E8|1.48519499E8|1.463171823333E8|
|2015-07-04|1.45438186E8|-1.4053438235062359|1.40993628E8| 1.4943842E8|    1.45290078E8|
|2015-07-05|1.51332605E8|-1.2183893416584186|1.45438186E8|1.40993628E8|    1.45921473E8|
+----------+------------+-------------------+------------+------------+----------------+
only showing top 5 rows



In [25]:
# for moving std deviation

from pyspark.sql.functions import stddev,round
window_MSD= Window.orderBy("Date").rowsBetween(-2,0)
df_centralized=df_centralized.withColumn("mov_stddev", round(stddev("total_load").over(window_MSD),4))
df_centralized.show(5)

26/04/10 13:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+------------+
|      Date|  total_load|            z_score|        lag1|        lag2|         Mov_Avg|  mov_stddev|
+----------+------------+-------------------+------------+------------+----------------+------------+
|2015-07-01|1.48519499E8|-1.3076131946985547|        NULL|        NULL|    1.48519499E8|        NULL|
|2015-07-02| 1.4943842E8|  -1.27846759113456|1.48519499E8|        NULL|   1.489789595E8| 649775.2705|
|2015-07-03|1.40993628E8|-1.5463127702222008| 1.4943842E8|1.48519499E8|1.463171823333E8|4633171.3767|
|2015-07-04|1.45438186E8|-1.4053438235062359|1.40993628E8| 1.4943842E8|    1.45290078E8|4224343.7319|
|2015-07-05|1.51332605E8|-1.2183893416584186|1.45438186E8|1.40993628E8|    1.45921473E8|5186403.9656|
+----------+------------+-------------------+------------+------------+----------------+------------+
only showing top 5 rows



In [26]:
# for trend
from pyspark.sql.functions import col

df_centralized=df_centralized.withColumn("trend", col("total_load")-col("lag1"))
df_centralized.show(5)

26/04/10 13:51:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+------------+----------+
|      Date|  total_load|            z_score|        lag1|        lag2|         Mov_Avg|  mov_stddev|     trend|
+----------+------------+-------------------+------------+------------+----------------+------------+----------+
|2015-07-01|1.48519499E8|-1.3076131946985547|        NULL|        NULL|    1.48519499E8|        NULL|      NULL|
|2015-07-02| 1.4943842E8|  -1.27846759113456|1.48519499E8|        NULL|   1.489789595E8| 649775.2705|  918921.0|
|2015-07-03|1.40993628E8|-1.5463127702222008| 1.4943842E8|1.48519499E8|1.463171823333E8|4633171.3767|-8444792.0|
|2015-07-04|1.45438186E8|-1.4053438235062359|1.40993628E8| 1.4943842E8|    1.45290078E8|4224343.7319| 4444558.0|
|2015-07-05|1.51332605E8|-1.2183893416584186|1.45438186E8|1.40993628E8|    1.45921473E8|5186403.9656| 5894419.0|
+----------+------------+-------------------+------------+------------+----------------+--------

In [27]:
df_centralized.selectExpr(
    "max(z_score)",
    "min(z_score)"
).show()

[Stage 105:===================================================>   (15 + 1) / 16]

+------------------+-------------------+
|      max(z_score)|       min(z_score)|
+------------------+-------------------+
|4.0288822384725895|-1.9898908848648427|
+------------------+-------------------+



In [28]:
#percent_rank = rank / (N - 1)

from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank
df_centralized=df_centralized.repartition(550)
window = Window.orderBy("z_score")

df_centralized = df_centralized.withColumn(
    "percentile",
    percent_rank().over(window)
)
df_centralized.show()

26/04/10 13:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+
|      Date|  total_load|            z_score|        lag1|        lag2|         Mov_Avg|     mov_stddev|       trend|          percentile|
+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+
|2015-08-01|1.27008217E8|-1.9898908848648427|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|
|2015-07-31|1.31522405E8|-1.8467134693292937|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|
|2015-07-25|1.33752964E8|-1.7759663775334804|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|
|2015-07-24|1.36192534E8| -1.698590041497396|1.43913454E8|1.49347436E8|1.431511413333E8|   6610499.3842|  -7720920.0| 0.00546448087431694|
|2015-07-30|1.36471572E8|-1

In [29]:
from pyspark.sql.functions import max
percentile=df_centralized.select(max("percentile")).show()

26/04/10 13:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:51:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+---------------+
|max(percentile)|
+---------------+
|            1.0|
+---------------+



In [30]:
# 0<percentile<0.4 --> Normal
# 0.4<percentile <0.75   -- degraded
# percentile >0.75    --- failure 

from pyspark.sql.functions import avg,when,col

df_centralized= df_centralized.withColumn("state_label",
                                          when(col("percentile")<=0.4 , 'Normal')
                                         .when((col("percentile")>0.4) & (col("percentile")<=0.75 ), 'Degraded')
                                         .when(col("percentile")>0.75, 'Failure')
                                        )
df_centralized.show()


26/04/10 13:52:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|      Date|  total_load|            z_score|        lag1|        lag2|         Mov_Avg|     mov_stddev|       trend|          percentile|state_label|
+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|2015-08-01|1.27008217E8|-1.9898908848648427|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|
|2015-07-31|1.31522405E8|-1.8467134693292937|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|
|2015-07-25|1.33752964E8|-1.7759663775334804|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|
|2015-07-24|1.36192534E8| -1.698590041497396|1.43913454E8|1.49347436E8|1.431511413333E8|   661

In [31]:
print("failure dates in centralized dataset are " ,df_centralized.filter(col("state_label") == "Failure").count())
print("degraded dates in centralized dataset are ",df_centralized.filter(col("state_label") == "Degraded").count())
print("Normal dates  centralized dataset are " ,df_centralized.filter(col("state_label") == "Normal").count())

26/04/10 13:52:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

failure dates in centralized dataset are  138


26/04/10 13:52:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

degraded dates in centralized dataset are  192


26/04/10 13:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Normal dates  centralized dataset are  220


In [32]:
df_centralized.columns

['Date',
 'total_load',
 'z_score',
 'lag1',
 'lag2',
 'Mov_Avg',
 'mov_stddev',
 'trend',
 'percentile',
 'state_label']

In [33]:
df_distributed.columns

['Page',
 'Date',
 'page_count',
 'Moving_avg',
 'Moving_std',
 'lag_1',
 'lag_2',
 'trend',
 'deviation',
 'z_score',
 'percentile',
 'state_label']

In [34]:
df_centralized = df_centralized.withColumnRenamed("total_load", "value") \
    .withColumnRenamed("lag1", "lag_1") \
    .withColumnRenamed("lag2", "lag_2") \
    .withColumnRenamed("Mov_Avg", "Moving_avg") \
    .withColumnRenamed("mov_stddev", "Moving_std")
df_centralized.show(5)

26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:52:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|      Date|       value|            z_score|       lag_1|       lag_2|      Moving_avg|     Moving_std|       trend|          percentile|state_label|
+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|2015-08-01|1.27008217E8|-1.9898908848648427|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|
|2015-07-31|1.31522405E8|-1.8467134693292937|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|
|2015-07-25|1.33752964E8|-1.7759663775334804|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|
|2015-07-24|1.36192534E8| -1.698590041497396|1.43913454E8|1.49347436E8|1.431511413333E8|   661

In [35]:
df_distributed = df_distributed.withColumnRenamed("page_count", "value")
df_distributed.show(5)

[Stage 198:>                                                        (0 + 1) / 1]

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|"""Weird_Al""_Yan...|2015-07-01|  144|     144.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|
|"""Weird_Al""_Yan...|2015-08-30|   20|     26.67|      5.77|   30|   30|-10.0|    -6.67|  -1.16|0.001824817518248...|     Normal|
|"""Weird_Al""_Yan...|2015-08-18|   19|     25.33|      5.51|   28|   29| -9.0|    -6.33|  -1.15|0.003649635036496...|     Normal|
|"""Weird_Al""_Yan...|2015-09-23|   40|      56.0|     13.89|   65|   63|-25.0|    -16.0|  -1.15|0.003649635036496...|     Normal|
|"""Weird_Al""_Yan...|2015-10-08|   38|      51.0|     11.27|   58|   57|-20.0|    

In [36]:
df_distributed.columns

['Page',
 'Date',
 'value',
 'Moving_avg',
 'Moving_std',
 'lag_1',
 'lag_2',
 'trend',
 'deviation',
 'z_score',
 'percentile',
 'state_label']

In [37]:
df_centralized.columns

['Date',
 'value',
 'z_score',
 'lag_1',
 'lag_2',
 'Moving_avg',
 'Moving_std',
 'trend',
 'percentile',
 'state_label']

In [38]:
df_centralized.write.mode("overwrite").parquet("/home/ubuntu/thesis/data/processed_data/system_state/centralized_data")

26/04/10 13:55:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:55:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:55:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:55:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:55:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 13:55:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/10 1

In [39]:
df_distributed.write.mode("overwrite").parquet("/home/ubuntu/thesis/data/processed_data/system_state/distributed_data")